# Asset Class Trend Following 策略回測 (equityNEW1)

本策略採用 2019-2023 年期間最佳化後的參數，對 **2024-2025** 年期間進行外樣本回測 (Out-of-sample backtest)。

In [ ]:
# 匯入所需套件
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


## 資料清洗
讀取清洗後的 `樣本集-1.xlsx` 檔案。

In [ ]:
# 定義資料清洗函數
def clean_data(filepath):
    df_prices = pd.read_excel(filepath, sheet_name='還原收盤價', header=None)
    df_volume = pd.read_excel(filepath, sheet_name='成交量', header=None)
    stock_codes = df_prices.iloc[0, 1:].values
    stock_names = df_prices.iloc[1, 1:].values
    date_strings = df_prices.iloc[2:, 0].astype(str).str[:8]
    dates = pd.to_datetime(date_strings, format='%Y%m%d')
    prices = df_prices.iloc[2:, 1:].astype(float)
    prices.index = dates
    prices.columns = stock_codes
    volumes = df_volume.iloc[2:, 1:].astype(float)
    volumes.index = dates
    volumes.columns = stock_codes
    code_to_name = dict(zip(stock_codes, stock_names))
    return prices, volumes, code_to_name

prices, volumes, code_to_name = clean_data('樣本集-1.xlsx')


## 回測引擎
包含 30M 成交金額濾網與 5/10/20 日均線強勢濾網。

In [ ]:
class BacktesterV2:
    def __init__(self, prices, volumes, code_to_name, initial_capital=30000000):
        self.prices_df = prices
        self.volumes_df = volumes
        self.prices = prices.values
        self.volumes = volumes.values
        self.dates = prices.index
        self.assets = prices.columns
        self.code_to_name = code_to_name
        self.initial_capital = initial_capital

    def run(self, sma_period, roc_period, stop_loss_pct, rebalance_interval=6, stop_loss_type='peak', ma_stop_period=5):
        sma = self.prices_df.rolling(window=sma_period).mean().values
        roc = self.prices_df.pct_change(periods=roc_period).values
        sma5 = self.prices_df.rolling(window=5).mean().values
        sma10 = self.prices_df.rolling(window=10).mean().values
        sma20 = self.prices_df.rolling(window=20).mean().values
        
        surplus_pool = float(self.initial_capital)
        slots = {0: None, 1: None, 2: None}
        equity_curve_data = [] 

        start_idx = max(sma_period, roc_period, 20)
        peak_equity = float(self.initial_capital)

        for i in range(start_idx, len(self.dates)):
            date = self.dates[i]
            current_prices = self.prices[i]
            stock_mv = 0.0
            for s_id, info in slots.items():
                if info: stock_mv += info['shares'] * current_prices[info['asset_idx']]
            
            total_equity = surplus_pool + stock_mv
            peak_equity = max(peak_equity, total_equity)
            drawdown = (total_equity - peak_equity) / peak_equity
            equity_curve_data.append({'日期': date, '權益': total_equity, '回撤(Drawdown)': drawdown})

            if i == len(self.dates) - 1: break
            next_prices = self.prices[i+1]

            # 停損
            for s_id, info in slots.items():
                if info:
                    a_idx = info['asset_idx']
                    curr_p = current_prices[a_idx]
                    if curr_p < info['max_price'] * (1 - stop_loss_pct):
                        sell_p = next_prices[a_idx]
                        surplus_pool += info['shares'] * sell_p * 0.995575
                        slots[s_id] = None

            # 再平衡
            if (i - start_idx) % rebalance_interval == 0:
                valid_signals = []
                sorted_idx = np.argsort(roc[i])[::-1]
                for idx in sorted_idx:
                    if len(valid_signals) >= 3: break
                    p = current_prices[idx]
                    amount = p * self.volumes[i][idx] * 1000
                    if p > sma[i][idx] and roc[i][idx] > 0 and amount > 30000000 and                        p > sma5[i][idx] and p > sma10[i][idx] and p > sma20[i][idx]:
                        valid_signals.append(idx)
                
                # 賣出與買入
                for s_id, info in list(slots.items()):
                    if info and info['asset_idx'] not in valid_signals:
                        surplus_pool += info['shares'] * next_prices[info['asset_idx']] * 0.995575
                        slots[s_id] = None
                
                for a_idx in valid_signals:
                    if a_idx not in [s['asset_idx'] for s in slots.values() if s]:
                        for s_id in slots:
                            if not slots[s_id]:
                                buy_p = next_prices[a_idx]
                                shares = (int(10000000 // (buy_p * 1.001425)) // 1000) * 1000
                                if shares > 0:
                                    surplus_pool -= shares * buy_p * 1.001425
                                    slots[s_id] = {'asset_idx': a_idx, 'shares': shares, 'max_price': buy_p}
                                break

        return pd.DataFrame(equity_curve_data)

def calculate_metrics(eq_df):
    eq = eq_df['權益']
    total_ret = (eq.iloc[-1] / eq.iloc[0]) - 1
    years = (eq_df['日期'].iloc[-1] - eq_df['日期'].iloc[0]).days / 365.25
    cagr = (1 + total_ret)**(1/years) - 1
    mdd = eq_df['回撤(Drawdown)'].min()
    return cagr, mdd, cagr/abs(mdd)


## 執行回測 (2024-2025)
參數：SMA=117, ROC=59, SL=10%, Reb=9。

In [ ]:
# 執行回測
bt = BacktesterV2(prices, volumes, code_to_name)
eq = bt.run(117, 59, 0.1, 9, 'peak', 20)

# 績效總結 (2024-2025)
mask = (eq['日期'] >= '2024-01-01') & (eq['日期'] <= '2025-12-31')
cagr, mdd, calmar = calculate_metrics(eq[mask])
print(f"年化報酬率 (CAGR): {cagr:.2%}")
print(f"最大回撤 (MaxDD): {mdd:.2%}")
print(f"Calmar Ratio: {calmar:.2f}")

# 權益曲線
plt.figure(figsize=(12, 6))
plt.plot(eq[mask]['日期'], eq[mask]['權益'])
plt.title('Equity Curve (2024-2025) - NEW1')
plt.grid(True)
plt.show()
